# BioChannel Notebook 5 — Edge Biology Assistant

**Create a small local-first dataset and lightweight simulation outputs.**

This notebook is part of the **BioChannel** Kaggle hackathon project. It is designed to be run inside Kaggle Notebooks and then reused by the Streamlit dashboard.

## How to use this notebook in BioChannel

1. Attach the relevant Kaggle dataset using **Add Data** in the Kaggle notebook sidebar.
2. Run the notebook end-to-end.
3. Save processed outputs into `/kaggle/working/processed/`.
4. Copy the exported CSV/JSON files into the BioChannel Streamlit app under `data/processed/`.
5. Reuse the helper functions in the app modules: `data_loader.py`, `preprocessing.py`, `simulation.py`, `info_theory.py`, and `prediction.py`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import mutual_info_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

np.random.seed(42)


In [ ]:
from pathlib import Path

INPUT_DIR = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
PROCESSED_DIR = WORKING_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

print('Input datasets available:')
for p in INPUT_DIR.glob('*'):
    print(' -', p)

## Dataset

Recommended dataset:

- Small gene expression dataset: `mehranksingh/gene-expression`

Science mapping:

This module demonstrates BioChannel in a lightweight setting: small data, small simulation, fast explanation, and optional local Gemma via Ollama/LiteRT.

In [ ]:
csv_files = list(INPUT_DIR.rglob('*.csv'))
for f in csv_files[:20]: print(f)

In [ ]:
def load_small_expression_or_demo(csv_files):
    for f in csv_files:
        try:
            df = pd.read_csv(f)
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if 5 <= len(numeric_cols) <= 300 and len(df) >= 10:
                print('Using:', f)
                return df
        except Exception:
            pass
    print('No suitable dataset found. Creating small demo dataset.')
    X = np.random.normal(0,1,(80,20))
    df = pd.DataFrame(X, columns=[f'GENE_{i:02d}' for i in range(20)])
    df['state'] = np.random.choice(['Baseline','Stress-like'], 80)
    return df

df = load_small_expression_or_demo(csv_files)
df.head()

In [ ]:
numeric = df.select_dtypes(include=[np.number]).copy().replace([np.inf,-np.inf], np.nan).fillna(0)
if numeric.min().min() >= 0:
    X = np.log1p(numeric)
else:
    X = numeric

# Keep a very small feature set for edge/local dashboard mode.
edge_features = X.var(axis=0).sort_values(ascending=False).head(min(25, X.shape[1])).index.tolist()
edge_df = X[edge_features].copy()
edge_df.to_csv(PROCESSED_DIR / 'edge_small_expression.csv', index=False)
pd.Series(edge_features).to_csv(PROCESSED_DIR / 'edge_features.csv', index=False, header=['feature'])
edge_df.head()

In [ ]:
def edge_decision(signal_strength=0.5, noise_level=0.2):
    proliferation = 0.8 * signal_strength - 0.3 * noise_level
    adaptation = 0.5 * signal_strength + 0.6 * noise_level
    uncertainty = noise_level
    scores = np.array([proliferation, adaptation, uncertainty])
    scores = scores - scores.max()
    probs = np.exp(scores) / np.exp(scores).sum()
    return dict(zip(['Proliferation-like','Stress-adaptation','Uncertain/noisy'], probs.round(4)))

edge_example = edge_decision()
with open(PROCESSED_DIR / 'edge_decision_example.json', 'w') as f:
    json.dump(edge_example, f, indent=2)
edge_example

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(edge_example.keys(), edge_example.values())
plt.ylim(0,1)
plt.title('Edge Biology Assistant: Lightweight Decision Output')
plt.xticks(rotation=20)
plt.show()

## Streamlit integration

Use:

- `edge_small_expression.csv` → small local demo data
- `edge_features.csv` → local explanation context
- `edge_decision_example.json` → simple local decision output

For hackathon positioning, this tab can mention Ollama, LiteRT, or local fallback mode. Keep the demo lightweight and accessible.